# Re-rankers testing

Here the different re-rankers of NLP4BIA library are tested.

## Retrieve candidates

Use, for example, a dense retriever to get the initial candidates.

In [4]:
import pandas as pd
from nlp4bia.datasets.benchmark.medprocner import MedprocnerLoader, MedprocnerGazetteer
from nlp4bia.linking.retrievers import DenseRetriever
from sentence_transformers import SentenceTransformer

model_name = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/biencoder_medprocner_1_epoch_32_batch_5_parents_stag"

st_model = SentenceTransformer(model_name)

df_proc = MedprocnerLoader().df
gaz_proc = MedprocnerGazetteer().df

ls_terms = gaz_proc["term"].tolist()
ls_mentions = df_proc["span"].tolist()[:10]

gaz_proc = gaz_proc.sort_values(by=["code", "mainterm"], ascending=[True, False])
vector_db = st_model.encode(ls_terms, 
                            show_progress_bar=True, 
                            convert_to_tensor=True, 
                            normalize_embeddings=True)

biencoder = DenseRetriever(vector_db=vector_db, model=st_model)

ls_medprocner_train = biencoder.retrieve_top_k(
                                                ls_mentions, 
                                                gaz_proc, 
                                                k=200, 
                                                input_format="text",
                                                return_documents=True
                                            )

Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 113.49it/s]


In [5]:
from sentence_transformers import CrossEncoder
from tqdm import tqdm
import numpy as np

ce_model_name = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/crossencoder_medprocner_5_epoch_16_batch"
ce_model = CrossEncoder(ce_model_name, device="cuda")

term2code = gaz_proc.set_index("term")["code"].to_dict()
ls_mentions = df_proc["span"].tolist()
ls_candidates = ls_medprocner_train
# 1) Build one big list of (mention, candidate) pairs, 
#    while keeping track of where each mention’s block starts/ends.
all_pairs  = []   # will hold tuples (mention, candidate_term)
offsets    = []   # will hold (start_idx, end_idx) for each mention
cursor     = 0

for mention, d_candidates in zip(ls_mentions, ls_candidates):
    candidates = d_candidates["terms"]
    if len(candidates) == 0:
        offsets.append((None, None))  # mark empty
        continue

    start = cursor
    # append one pair for each candidate of this mention
    for cand in candidates:
        all_pairs.append((mention, cand))
        cursor += 1
    end = cursor       # exclusive
    offsets.append((start, end))

# 2) Predict all scores in batches
# The CrossEncoder.predict() method returns a single float per pair.
# By default batch_size=32, but you can increase if GPU memory allows (e.g. 128 or 256).
scores = ce_model.predict(all_pairs, batch_size=4096, show_progress_bar=True)

# 3) Now re‐group and sort each mention’s candidates by score

ls_reranked_cands = []
for (mention, d_candidates), (start, end) in zip(zip(df_proc["span"].tolist(), ls_medprocner_train), offsets):
    d_reranked_cands = {}
    
    # slice out this mention’s score vector
    this_scores      = scores[start:end]                 # shape = (num_cands_for_this_mention,)
    this_candidates  = d_candidates["terms"]              # length = same
    this_codes       = [term2code[t] for t in this_candidates]

    # sort by score descending
    sorted_idx = np.argsort(this_scores)[::-1]
    reranked_terms  = [ this_candidates[i] for i in sorted_idx ]
    reranked_codes  = [ this_codes[i]      for i in sorted_idx ]
    reranked_scores = [ float(this_scores[i])   for i in sorted_idx ]

    d_reranked_cands["mention"] = mention
    d_reranked_cands["terms"]   = reranked_terms
    d_reranked_cands["codes"]   = reranked_codes
    d_reranked_cands["scores"]  = reranked_scores
    
    ls_reranked_cands.append(d_reranked_cands)

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.32it/s]


In [6]:
len(ls_reranked_cands)

10

In [9]:
from nlp4bia.linking.rerankers import CrossEncoderReranker

# Example:
ce_model_path = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/crossencoder_medprocner_5_epoch_16_batch"
reranker = CrossEncoderReranker(
                                    model_path=ce_model_path,
                                    device="cuda",
                                    batch_size=4096,
                                    term2code=term2code,
                                    show_progress_bar=True
                                )

# Run reranking:
ls_reranked = reranker.rerank(ls_mentions[:10], ls_medprocner_train[:10], return_documents=True)


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 14.40it/s]


Test they are the same

In [10]:
print(ls_reranked[0]["mention"] == ls_reranked_cands[0]["mention"])
print(ls_reranked[0]["terms"] == ls_reranked_cands[0]["terms"])
print(ls_reranked[0]["codes"] == ls_reranked_cands[0]["codes"])
print(np.abs(np.array(ls_reranked[0]["scores"]) - np.array(ls_reranked_cands[0]["scores"])).sum() < 1e-5)

True
True
True
True
